# Data Analyst Assessment — Sample Superstore

**Dataset:** Sample - Superstore  
**Assessment:** VirtuBox Infotech — Data Analyst Assessment  
**Purpose:** Analyze sales and profitability to identify business opportunities, performance gaps and operational improvements.

This notebook answers Q1–Q10 using Python, Pandas, NumPy, Matplotlib and Seaborn.

> The assessment asks for one large publicly available dataset, a defined business problem, data processing, at least five insights, an unusual result, data-quality limitations, recommendations, a Looker Studio dashboard, a 5–7 slide management presentation, AI-use disclosure, and final submission documentation.


## 0. Setup

Place `Sample - Superstore.csv` in the same folder as this notebook. If your filename/path is different, change `DATA_PATH`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "Sample - Superstore.csv"

# The provided file contains a non-UTF8 character, so latin1 is used.
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Shape:", df.shape)
display(df.head())


# Q1 — Dataset Selection

**Dataset name:** Sample - Superstore

**Source:** Tableau Public Sample Data  
https://public.tableau.com/app/resources/sample-data

**Why this dataset:** Tableau describes Superstore as a sample source containing dates, geography, product hierarchy, sales and profit. It is transaction-level data with enough dimensions and measures for meaningful cleaning, segmentation and business analysis.

**Important note:** The source file contains 9,994 rows, so it does not meet the optional 50,000+ row criterion. However, the assessment also allows a dataset with sufficient complexity for meaningful processing and analysis. This dataset meets that criterion through transaction-level detail, multiple dimensions, dates, geography, product hierarchy, discounts and positive/negative profit.


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)


# Q2 — Business Problem, Questions and Hypotheses

### A. Business problem
How can management grow sales while protecting profit, especially by reducing margin leakage from discounts and low-profit product areas?

### B. Questions
1. Which categories and sub-categories generate the most sales and profit, and where are the largest profitability gaps?
2. How does discount level relate to profit margin and loss-making transactions?
3. Which regions and customer segments combine scale with attractive profitability?
4. How have sales and profit changed over time?
5. Which products/sub-categories create disproportionate profit or loss?

### C. Hypotheses
- **H1:** Higher discount levels are associated with lower profit margins.
- **H2:** Technology and Office Supplies have stronger profitability than Furniture.


# Q3 — Data Processing

Cleaning decisions:
1. Parse Order Date and Ship Date as dates.
2. Check and remove exact duplicates if present.
3. Check missing values and invalid ranges.
4. Create calculated fields: Ship Days, Year, Month, Year-Month, Profit Margin, Profit per Unit, Order Sales.
5. Categorize discount levels and profitability.
6. Use unique Order ID for order-level metrics because rows are line items.
7. Flag statistical outliers with the IQR method but do not automatically delete them.


In [ ]:
data = df.copy()

# Dates
data["Order Date"] = pd.to_datetime(data["Order Date"], format="%m/%d/%Y")
data["Ship Date"] = pd.to_datetime(data["Ship Date"], format="%m/%d/%Y")

# Calculated fields
data["Ship Days"] = (data["Ship Date"] - data["Order Date"]).dt.days
data["Year"] = data["Order Date"].dt.year
data["Month"] = data["Order Date"].dt.month
data["Month Name"] = data["Order Date"].dt.strftime("%b")
data["Year-Month"] = data["Order Date"].dt.to_period("M").astype(str)
data["Profit Margin"] = data["Profit"] / data["Sales"]
data["Profit per Unit"] = data["Profit"] / data["Quantity"]
data["Profitability"] = np.where(data["Profit"] < 0, "Loss", "Profit")
data["Discount Band"] = pd.cut(
    data["Discount"],
    bins=[-0.001, 0, 0.10, 0.20, 0.30, 1],
    labels=["0%", "1-10%", "11-20%", "21-30%", ">30%"]
)
data["Order Sales"] = data.groupby("Order ID")["Sales"].transform("sum")

# Quality checks
print("Exact duplicate rows:", data.duplicated().sum())
print("Total missing values:", int(data.isna().sum().sum()))
print("Ship date before order date:", int((data["Ship Date"] < data["Order Date"]).sum()))
print("Invalid discounts:", int(((data["Discount"] < 0) | (data["Discount"] > 1)).sum()))
print("Non-positive quantities:", int((data["Quantity"] <= 0).sum()))
print("Non-positive sales:", int((data["Sales"] <= 0).sum()))

display(data.head())


In [ ]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return int(mask.sum()), lower, upper

for col in ["Sales", "Profit", "Discount", "Ship Days"]:
    count, lower, upper = iqr_outliers(data[col])
    print(f"{col}: {count:,} IQR outliers | lower={lower:.2f}, upper={upper:.2f}")


### Q3 — Why these decisions matter

- **Date conversion:** without it, time trends and shipping-duration analysis are unreliable.
- **Duplicate check:** duplicate rows could double-count revenue and profit.
- **Missing/invalid checks:** invalid values can bias totals and ratios.
- **Calculated fields:** management needs margin, shipping time and time-period metrics, not just raw columns.
- **Correct grain:** a row is a transaction line, not necessarily a whole order. Counting rows as orders would overstate order volume.
- **Outliers:** extreme transactions are retained because they may be genuine business events; they are flagged for review instead of deleted automatically.


# Q4 — Exploratory & Descriptive Analysis

The following cells calculate the evidence used in the management insights.


In [ ]:
# Overall KPIs
total_sales = data["Sales"].sum()
total_profit = data["Profit"].sum()
overall_margin = total_profit / total_sales
orders = data["Order ID"].nunique()
customers = data["Customer ID"].nunique()
aov = data.groupby("Order ID")["Sales"].sum().mean()
loss_rows = int((data["Profit"] < 0).sum())

kpis = pd.Series({
    "Total Sales": total_sales,
    "Total Profit": total_profit,
    "Profit Margin": overall_margin,
    "Unique Orders": orders,
    "Unique Customers": customers,
    "Average Order Value": aov,
    "Loss-making Rows": loss_rows,
    "Loss-making Row %": loss_rows / len(data),
    "Average Ship Days": data["Ship Days"].mean()
})
display(kpis)


In [ ]:
# Category analysis
category = data.groupby("Category").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order ID","nunique")
)
category["Profit Margin"] = category["Profit"] / category["Sales"]
category["Sales Share"] = category["Sales"] / total_sales
category["Profit Share"] = category["Profit"] / total_profit
display(category.sort_values("Profit", ascending=False).round(4))


In [ ]:
# Sub-category analysis
subcategory = data.groupby("Sub-Category").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Quantity=("Quantity","sum"),
    Rows=("Sales","size")
)
subcategory["Profit Margin"] = subcategory["Profit"] / subcategory["Sales"]
display(subcategory.sort_values("Profit").round(3))


In [ ]:
# Discount analysis
discount = data.groupby("Discount Band", observed=True).agg(
    Rows=("Sales","size"),
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    AvgDiscount=("Discount","mean")
)
discount["Profit Margin"] = discount["Profit"] / discount["Sales"]
display(discount.round(3))

print("Correlation: Discount vs Profit Margin =", round(
    data["Discount"].corr(data["Profit Margin"]), 3
))


In [ ]:
# Region and segment analysis
region = data.groupby("Region").agg(Sales=("Sales","sum"), Profit=("Profit","sum"), Rows=("Sales","size"))
region["Profit Margin"] = region["Profit"] / region["Sales"]

segment = data.groupby("Segment").agg(
    Sales=("Sales","sum"), Profit=("Profit","sum"), Rows=("Sales","size"),
    Orders=("Order ID","nunique"), Customers=("Customer ID","nunique")
)
segment["Profit Margin"] = segment["Profit"] / segment["Sales"]

display(region.sort_values("Profit", ascending=False).round(3))
display(segment.round(3))


In [ ]:
# Yearly trend
year = data.groupby("Year").agg(Sales=("Sales","sum"), Profit=("Profit","sum"), Rows=("Sales","size"))
year["Profit Margin"] = year["Profit"] / year["Sales"]
year["Sales Growth %"] = year["Sales"].pct_change() * 100
year["Profit Growth %"] = year["Profit"].pct_change() * 100
display(year.round(3))


In [ ]:
# Visual 1: category sales vs profit
plt.figure(figsize=(8,5))
x = np.arange(len(category))
plt.bar(x-0.18, category["Sales"], width=0.36, label="Sales")
plt.bar(x+0.18, category["Profit"], width=0.36, label="Profit")
plt.xticks(x, category.index)
plt.title("Sales vs Profit by Category")
plt.ylabel("Amount")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Visual 2: discount band margin
plt.figure(figsize=(8,5))
plt.bar(discount["Discount Band"].astype(str), discount["Profit Margin"]*100)
plt.axhline(0, linewidth=1)
plt.title("Profit Margin by Discount Band")
plt.ylabel("Profit Margin (%)")
plt.xlabel("Discount Band")
plt.tight_layout()
plt.show()


In [ ]:
# Visual 3: annual sales and profit
plt.figure(figsize=(8,5))
plt.plot(year.index, year["Sales"], marker="o", label="Sales")
plt.plot(year.index, year["Profit"], marker="o", label="Profit")
plt.title("Annual Sales and Profit Trend")
plt.ylabel("Amount")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


### Five management insights

**Insight 1 — Furniture has high sales but weak profitability.**  
Furniture contributes about 32.3% of sales but only about 6.4% of total profit, with an overall margin of about 2.5%. Tables are loss-making.

**Insight 2 — High discounts are strongly associated with margin erosion.**  
The correlation between discount and row-level profit margin is about -0.864. The >30% discount band has a negative aggregate margin of about -48.2%.

**Insight 3 — Regional profitability varies materially.**  
West has the strongest margin at about 14.9%, while Central is lowest at about 7.9%.

**Insight 4 — Home Office is a smaller but more profitable segment.**  
Home Office has the highest segment margin at about 14.0%, versus about 11.5% for Consumer.

**Insight 5 — Sales growth is strong, but margin needs monitoring.**  
Sales grew from about $484K in 2014 to $733K in 2017. Profit also grew, but margin slipped from about 13.4% in 2016 to about 12.7% in 2017.

These are associations and descriptive findings; they do not prove causation.


# Q5 — Surprising / Unexpected Result

### Finding
I initially expected a high-sales category to contribute a broadly similar share of profit.

### What happened?
Furniture generated roughly **32.3% of sales but only 6.4% of total profit**. Its margin was only about **2.5%**, while Technology was about **17.4%**.

### Why might this happen?
The data shows loss-making Furniture sub-categories, especially Tables, and a strong relationship between higher discount levels and lower margins. However, the dataset does not contain enough cost/pricing detail to prove the exact causal mechanism.

### Additional analysis
I compared category and sub-category margins, discount bands, loss-making transactions and the discount–profit-margin relationship.

### Conclusion
Furniture should be treated as a profitability-priority area. Management should investigate pricing, discounting, product costs and product mix before pursuing additional volume.


# Q6 — Data Quality, Limitations and Risks

### Data-quality / analytical risks
1. **Repeated Order IDs:** multiple rows can belong to one order. I use `nunique(Order ID)` for order counts.
2. **Statistical outliers:** high Sales/Profit values exist. I flag them with IQR but retain them because they may be legitimate.
3. **Historical/fictitious data:** this is a Tableau sample dataset covering 2014–2017, not current company data.
4. **Confounding:** discount is related to margin, but product mix, cost and transaction characteristics may also affect profit.

### Limitations
- No detailed cost, inventory, competitor, campaign or customer-acquisition variables are available.
- The sample may not represent current market conditions or a B2B technology company's actual economics.

### Conclusion that cannot safely be made
We cannot safely say that high discounts **caused** losses. The data supports a strong association, not causal proof.


# Q7 — Three Actionable Recommendations

### 1. Tighten discount governance — High priority
- **What:** Add approval thresholds for discounts above 20%.
- **Supports:** Discount-band analysis.
- **Who:** Sales leadership, Pricing and Finance.
- **Outcome:** Lower margin leakage.
- **Measure:** Profit margin by discount band, share of sales discounted >20%, total profit.

### 2. Run a Furniture profitability recovery plan — High priority
- **What:** Review pricing, discounts, costs and product mix, prioritizing Tables.
- **Supports:** Furniture's low margin and loss-making Tables.
- **Who:** Category Management, Pricing and Finance.
- **Outcome:** Higher Furniture margin and lower avoidable losses.
- **Measure:** Furniture margin, Tables profit, loss-making transaction rate.

### 3. Scale profitable Home Office opportunities and improve Central — Medium priority
- **What:** Target profitable Home Office customers/products and conduct a Central-region profitability review.
- **Supports:** Home Office's high margin and Central's low margin.
- **Who:** Sales/Marketing and Regional leadership.
- **Outcome:** Better profit mix and regional economics.
- **Measure:** Home Office sales/profit growth and Central profit margin.


# Q8 — Looker Studio Dashboard

The assessment requires an **interactive Google Looker Studio dashboard**. This notebook prepares the analysis-ready data and defines the dashboard.

### Recommended dashboard
- KPI cards: Sales, Profit, Margin, Orders, Customers.
- Annual/monthly Sales & Profit trend.
- Category/Sub-category Sales vs Profit.
- Region Sales and Profit Margin comparison.
- Discount-band Profit and Profit Margin.
- Filters: Year, Region, Category, Segment, Ship Mode.

**Important:** An actual shareable Looker Studio link cannot be created from this notebook because it requires access to the user's Google account and Drive/Looker Studio workspace. The supplied `dashboard_mockup.png` is only a design reference; it is not a substitute for the required interactive dashboard.


# Presentation (5–7 slides)

Use the supplied PowerPoint file for the management presentation.

1. Business Problem
2. Data & Methodology
3. Key Findings
4. Deep-Dive Insight
5. Recommendations
6. Expected Business Impact
7. Limitations & Next Steps


# Q10 — How I Used AI

**AI tool:** ChatGPT.

**Uses:** Understanding the assessment, planning the analysis, generating/debugging Python, structuring business questions, interpreting results, drafting documentation and visualization ideas.

**Example where AI helped:** It helped structure a reproducible Q1–Q10 workflow and identify useful profitability/discount analyses.

**Verification example:** Numerical outputs were independently calculated from the provided CSV using Python before being included in the submission. In particular, dataset size, missing/duplicate checks, category margins, discount-band profitability and KPI totals were verified.


In [ ]:
# Export the processed dataset for Google Sheets / Looker Studio
data.to_csv("Processed_Data.csv", index=False)

# Export a compact dashboard dataset
dashboard = data[[
    "Order ID","Order Date","Year","Month","Region","Segment","Category",
    "Sub-Category","Ship Mode","Sales","Quantity","Discount","Profit",
    "Profit Margin","Profitability"
]].copy()
dashboard.to_csv("Dashboard_Data.csv", index=False)

print("Created: Processed_Data.csv and Dashboard_Data.csv")


# Final checklist

- [x] Q1 dataset, source, size, description and business use
- [x] Q2 business problem, questions and hypotheses
- [x] Q3 processing and cleaning
- [x] Q4 at least five business insights
- [x] Q5 surprising result with additional analysis and caveat
- [x] Q6 data-quality issues, limitations and unsafe conclusion
- [x] Q7 three prioritized recommendations
- [ ] Q8 actual interactive Looker Studio dashboard + shareable link — **must be created in your Google account**
- [x] Q8 dashboard design / mockup
- [x] 7-slide presentation content
- [x] Q10 AI-use disclosure
- [x] Supporting processed data

## Final submission folder
Place the notebook, the Google Sheet (converted from the supplied Excel workbook), the presentation, README/methodology, and supporting data in one Google Drive folder.
